# From Jira to Pull Request: Building an AI Bug-Fix Agent

Companion notebook for the [blog post](https://ai-terminal.net/llm/ai-engineering/code-gen/2026/04/08/jira-to-pr/).

This notebook implements four core building blocks of the AI bug-fix pipeline:
1. **Code indexing with tree-sitter**: parse Python source, extract functions, chunk at AST boundaries
2. **Hybrid retrieval**: vector search (OpenAI embeddings) + BM25 keyword search + reciprocal rank fusion
3. **Dependency graph**: extract call relationships from the AST for "who calls X?" queries
4. **A simplified ReAct agent loop**: process a sample bug ticket against an indexed repo

You'll need an OpenAI API key for embedding and the agent loop.

In [ ]:
!pip install -q tree-sitter tree-sitter-python openai rank-bm25 tiktoken numpy

In [1]:
import os

# Set your OpenAI API key
# os.environ["OPENAI_API_KEY"] = "sk-..."
if "OPENAI_API_KEY" not in os.environ:
    try:
        from google.colab import userdata
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except (ImportError, Exception):
        print("No API key found. Code indexing works without it. Embedding and agent loop need one.")

No API key found. Code indexing works without it. Embedding and agent loop need one.


## Part 1: Code Indexing with tree-sitter

We'll parse Python source code into its AST and extract every function definition as a structured record.

In [ ]:
import tree_sitter_python as tspython
from tree_sitter import Language, Parser

PY_LANGUAGE = Language(tspython.language())
parser = Parser(PY_LANGUAGE)
print(f"Parser ready: {PY_LANGUAGE}")

### Sample Repository

We'll create a small sample codebase to index. This simulates a Python service with a bug: `UserService.get_profile()` doesn't handle `None` from a cache miss.

In [ ]:
SAMPLE_REPO = {
    "src/services/user_service.py": '''from src.cache import UserCache
from src.models import User


class UserService:
    """Handles user profile operations."""

    def __init__(self, cache: UserCache):
        self.cache = cache

    def get_profile(self, user_id: str) -> User:
        """Returns the cached profile for a user ID.
        Returns None if the user is not in cache."""
        user = self.cache.get(user_id)
        return user

    def update_profile(self, user_id: str, data: dict) -> User:
        """Updates a user profile and refreshes the cache."""
        user = self.get_profile(user_id)
        user.name = data.get("name", user.name)
        user.email = data.get("email", user.email)
        self.cache.set(user_id, user)
        return user
''',
    "src/api/user_controller.py": '''from src.services.user_service import UserService


class UserController:
    """HTTP handler for user endpoints."""

    def __init__(self, user_service: UserService):
        self.user_service = user_service

    def get_user(self, request):
        """GET /users/:id - returns user profile."""
        user_id = request.path_params["id"]
        profile = self.user_service.get_profile(user_id)
        return {"status": 200, "body": profile.to_dict()}

    def list_users(self, request):
        """GET /users - returns all cached users."""
        return {"status": 200, "body": self.user_service.list_all()}
''',
    "src/cache.py": '''import time


class UserCache:
    """Simple in-memory cache with TTL."""

    def __init__(self, ttl_seconds: int = 300):
        self._store = {}
        self._ttl = ttl_seconds

    def get(self, key: str):
        """Returns cached value or None if expired/missing."""
        entry = self._store.get(key)
        if entry is None:
            return None
        if time.time() - entry["ts"] > self._ttl:
            del self._store[key]
            return None
        return entry["value"]

    def set(self, key: str, value):
        """Stores a value with the current timestamp."""
        self._store[key] = {"value": value, "ts": time.time()}
''',
    "src/models.py": '''class User:
    """User model."""

    def __init__(self, user_id: str, name: str, email: str):
        self.user_id = user_id
        self.name = name
        self.email = email

    def to_dict(self):
        """Serialize to dictionary."""
        return {
            "user_id": self.user_id,
            "name": self.name,
            "email": self.email,
        }
''',
}

print(f"Sample repo: {len(SAMPLE_REPO)} files")
for path in SAMPLE_REPO:
    print(f"  {path}")

### Extracting Symbols from the AST

Walk the tree-sitter parse tree and extract every function and class definition as a structured record.

In [ ]:
def _walk(node):
    """Recursively yield all nodes in the tree."""
    yield node
    for child in node.children:
        yield from _walk(child)


def _get_docstring(body_node):
    """Extract docstring from a function/class body if present."""
    if body_node is None:
        return ""
    for child in body_node.children:
        if child.type == "expression_statement":
            expr = child.children[0] if child.children else None
            if expr and expr.type == "string":
                text = expr.text.decode()
                # Strip triple quotes
                for q in ['"""', "'''"]:
                    if text.startswith(q) and text.endswith(q):
                        return text[3:-3].strip()
                return text.strip('"\'')
    return ""

In [ ]:
def _get_class_for_method(node):
    """Walk up the tree to find the enclosing class name."""
    parent = node.parent
    while parent is not None:
        if parent.type == "class_definition":
            name_node = parent.child_by_field_name("name")
            if name_node:
                return name_node.text.decode()
        parent = parent.parent
    return None


def _extract_function(node, file_path: str) -> dict:
    name_node = node.child_by_field_name("name")
    params_node = node.child_by_field_name("parameters")
    body_node = node.child_by_field_name("body")
    docstring = _get_docstring(body_node)
    class_name = _get_class_for_method(node)

    return {
        "symbol_type": "method" if class_name else "function",
        "symbol_name": name_node.text.decode(),
        "class_name": class_name,
        "file_path": file_path,
        "start_line": node.start_point[0] + 1,
        "end_line": node.end_point[0] + 1,
        "parameters": params_node.text.decode() if params_node else "",
        "docstring": docstring,
        "full_source": node.text.decode(),
    }

In [ ]:
def extract_symbols(source: str, file_path: str) -> list[dict]:
    """Parse source and extract all function/method definitions."""
    tree = parser.parse(source.encode())
    symbols = []

    for node in _walk(tree.root_node):
        if node.type == "function_definition":
            symbols.append(_extract_function(node, file_path))

    return symbols

In [ ]:
# Index the entire sample repo
all_symbols = []
for path, source in SAMPLE_REPO.items():
    symbols = extract_symbols(source, path)
    all_symbols.extend(symbols)
    print(f"{path}: {len(symbols)} symbols")

print(f"\nTotal: {len(all_symbols)} symbols extracted")
for s in all_symbols:
    prefix = f"{s['class_name']}." if s['class_name'] else ""
    print(f"  {s['symbol_type']:8s} {prefix}{s['symbol_name']}  ({s['file_path']}:{s['start_line']}-{s['end_line']})")

### Chunking

Each symbol becomes one chunk. Large functions (>400 tokens) would be split into overlapping sub-chunks. Our sample functions are small enough that each is a single chunk.

In [ ]:
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4o")
MAX_CHUNK_TOKENS = 400
OVERLAP_TOKENS = 80


def chunk_symbol(symbol: dict) -> list[dict]:
    source = symbol["full_source"]
    token_count = len(enc.encode(source))

    if token_count <= MAX_CHUNK_TOKENS:
        chunk_id = f"{symbol['file_path']}::{symbol.get('class_name', '')}::{symbol['symbol_name']}::0"
        return [{**symbol, "chunk_id": chunk_id, "token_count": token_count}]

    # Split large functions into overlapping sub-chunks
    tokens = enc.encode(source)
    chunks = []
    start = 0
    idx = 0
    while start < len(tokens):
        end = min(start + MAX_CHUNK_TOKENS, len(tokens))
        chunk_text = enc.decode(tokens[start:end])
        chunk_id = f"{symbol['file_path']}::{symbol.get('class_name', '')}::{symbol['symbol_name']}::{idx}"
        chunk = {**symbol, "full_source": chunk_text, "chunk_id": chunk_id, "token_count": end - start}
        chunks.append(chunk)
        start += MAX_CHUNK_TOKENS - OVERLAP_TOKENS
        idx += 1

    return chunks


all_chunks = []
for symbol in all_symbols:
    all_chunks.extend(chunk_symbol(symbol))

print(f"{len(all_chunks)} chunks from {len(all_symbols)} symbols")
for c in all_chunks:
    print(f"  {c['chunk_id']}  ({c['token_count']} tokens)")

### Enrichment: Building the Embedding Input

The key insight: embed an enriched natural language + code string, not raw source code. This bridges the gap between ticket queries ("AttributeError when fetching user profile") and code (`.get(user_id)` returning `None`).

In [ ]:
def build_embedding_input(chunk: dict) -> str:
    parts = [f"File: {chunk['file_path']}"]

    if chunk.get("class_name"):
        parts.append(f"Class: {chunk['class_name']}")

    kind = chunk["symbol_type"].title()
    parts.append(f"{kind}: {chunk['symbol_name']}{chunk.get('parameters', '')}")

    if chunk.get("docstring"):
        parts.append(f"\nSummary: {chunk['docstring']}")

    parts.append(f"\nSource:\n{chunk['full_source']}")
    return "\n".join(parts)


# Show what the enriched input looks like for get_profile
target = next(c for c in all_chunks if c["symbol_name"] == "get_profile")
enriched = build_embedding_input(target)
print("=== Enriched embedding input ===")
print(enriched)

## Part 2: Hybrid Retrieval

We combine two search strategies:
- **Vector search** (OpenAI embeddings + cosine similarity) for semantic matching
- **BM25 keyword search** for exact identifier matching

Results are fused using Reciprocal Rank Fusion (RRF).

In [ ]:
import numpy as np


def cosine_sim(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

### Embed All Chunks

Send the enriched text strings to `text-embedding-3-large` and store the vectors alongside each chunk.

In [ ]:
import openai


def embed_chunks(chunks: list[dict]) -> list[dict]:
    texts = [build_embedding_input(c) for c in chunks]

    response = openai.embeddings.create(
        model="text-embedding-3-large",
        input=texts,
    )

    for chunk, item in zip(chunks, response.data):
        chunk["embedding"] = item.embedding

    return chunks


if "OPENAI_API_KEY" in os.environ:
    all_chunks = embed_chunks(all_chunks)
    print(f"Embedded {len(all_chunks)} chunks")
    print(f"Embedding dimensions: {len(all_chunks[0]['embedding'])}")
else:
    print("Skipped: set OPENAI_API_KEY to embed chunks")

### BM25 Index

Build a BM25 index over the source code. This handles exact identifier matching: when the agent searches for `get_profile`, BM25 finds the literal match.

In [ ]:
import re
from rank_bm25 import BM25Okapi


def tokenize_code(text: str) -> list[str]:
    """Split code into tokens: split on whitespace and punctuation,
    also split camelCase and snake_case into parts."""
    # Split on non-alphanumeric
    tokens = re.findall(r'[a-zA-Z_][a-zA-Z0-9_]*', text)
    expanded = []
    for t in tokens:
        # Split camelCase
        parts = re.findall(r'[a-z]+|[A-Z][a-z]*|[A-Z]+(?=[A-Z]|$)', t)
        expanded.append(t.lower())
        for p in parts:
            if p.lower() != t.lower():
                expanded.append(p.lower())
    return expanded


# Build the BM25 index
corpus = [tokenize_code(c["full_source"]) for c in all_chunks]
bm25_index = BM25Okapi(corpus)

# Test it: search for "get_profile"
query_tokens = tokenize_code("get_profile cache")
scores = bm25_index.get_scores(query_tokens)
print("BM25 scores for 'get_profile cache':")
for i, (chunk, score) in enumerate(zip(all_chunks, scores)):
    if score > 0:
        prefix = f"{chunk.get('class_name', '')}." if chunk.get('class_name') else ""
        print(f"  {score:.2f}  {prefix}{chunk['symbol_name']}  ({chunk['file_path']})")

### Reciprocal Rank Fusion

Combine vector and BM25 results without needing to normalize scores across different systems.

In [ ]:
def _to_ranks(scores: dict) -> dict:
    """Convert score dict to rank dict (1-based)."""
    sorted_ids = sorted(scores, key=scores.get, reverse=True)
    return {chunk_id: rank + 1 for rank, chunk_id in enumerate(sorted_ids)}


def reciprocal_rank_fusion(
    scores_a: dict, scores_b: dict, top_k: int = 5, k: int = 60
) -> list[tuple[str, float]]:
    """Fuse two ranked lists using RRF."""
    rank_a = _to_ranks(scores_a)
    rank_b = _to_ranks(scores_b)
    all_ids = set(rank_a) | set(rank_b)

    fused = {}
    for chunk_id in all_ids:
        rrf = 0.0
        if chunk_id in rank_a:
            rrf += 1.0 / (k + rank_a[chunk_id])
        if chunk_id in rank_b:
            rrf += 1.0 / (k + rank_b[chunk_id])
        fused[chunk_id] = rrf

    ranked = sorted(fused.items(), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]

In [ ]:
def hybrid_search(query: str, chunks: list[dict], top_k: int = 5):
    """Run vector + BM25 search and fuse results."""
    # Vector search
    q_emb = openai.embeddings.create(
        model="text-embedding-3-large", input=[query]
    ).data[0].embedding

    vec_scores = {
        c["chunk_id"]: cosine_sim(q_emb, c["embedding"])
        for c in chunks
        if "embedding" in c
    }

    # BM25 search
    bm25_corpus = [tokenize_code(c["full_source"]) for c in chunks]
    bm25 = BM25Okapi(bm25_corpus)
    bm25_raw = bm25.get_scores(tokenize_code(query))
    bm25_scores = {chunks[i]["chunk_id"]: bm25_raw[i] for i in range(len(chunks))}

    # Fuse
    fused = reciprocal_rank_fusion(vec_scores, bm25_scores, top_k)

    # Return full chunk data
    chunk_map = {c["chunk_id"]: c for c in chunks}
    results = []
    for chunk_id, score in fused:
        c = chunk_map[chunk_id]
        results.append({
            "chunk_id": chunk_id,
            "symbol_name": c["symbol_name"],
            "class_name": c.get("class_name"),
            "file_path": c["file_path"],
            "start_line": c["start_line"],
            "end_line": c["end_line"],
            "full_source": c["full_source"],
            "rrf_score": score,
        })
    return results

In [ ]:
# Test hybrid search with a ticket-style query
if "OPENAI_API_KEY" in os.environ:
    query = "AttributeError when fetching user profile from cache"
    results = hybrid_search(query, all_chunks, top_k=5)

    print(f"Query: '{query}'\n")
    for i, r in enumerate(results):
        prefix = f"{r['class_name']}." if r['class_name'] else ""
        print(f"{i+1}. {prefix}{r['symbol_name']}  ({r['file_path']}:{r['start_line']})  RRF={r['rrf_score']:.4f}")
else:
    print("Skipped: set OPENAI_API_KEY to run hybrid search")

### Dependency Graph

Embedding search answers "what code is semantically related to this ticket?" but can't answer "who calls `get_profile`?"
Those are graph traversal questions. We extract call relationships from the same tree-sitter parse and store them
so the agent can query callers and dependencies with a single lookup.

In [ ]:
def extract_calls(node, file_path: str, class_name: str = None) -> list[str]:
    """Walk a function body and extract raw call target strings."""
    calls = []
    for child in _walk(node):
        if child.type == "call":
            func = child.child_by_field_name("function")
            if func is not None:
                calls.append(func.text.decode())
    return calls


# Test: extract calls from UserController.get_user
src = SAMPLE_REPO["src/api/user_controller.py"]
tree = parser.parse(src.encode())
for node in _walk(tree.root_node):
    if node.type == "function_definition":
        name = node.child_by_field_name("name").text.decode()
        calls = extract_calls(node, "src/api/user_controller.py")
        if calls:
            print(f"{name}() calls: {calls}")

## Part 3: The ReAct Agent Loop

A simplified version of the agentic harness. The agent has access to three skills:
- `search_code(query)` — hybrid retrieval over the indexed repo
- `read_file(path, start_line, end_line)` — read source from the sample repo
- `think(reasoning)` — extended reasoning (logged but not executed)

The agent processes a bug ticket and produces a diagnosis with a fix strategy.

The raw call strings like `self.user_service.get_profile` need to be resolved to symbol names.
We use a simple heuristic: strip `self.`, match the final method name against known symbols.

In [ ]:
def resolve_call(raw_call: str, all_symbols: list[dict]) -> str | None:
    """Resolve a raw call expression to a known symbol name.
    Strip self/cls prefix, then match the final method name against the symbol table."""
    call = raw_call
    for prefix in ["self.", "cls."]:
        if call.startswith(prefix):
            call = call[len(prefix):]

    # Get the final method name (last element after dots)
    parts = call.split(".")
    method_name = parts[-1]

    # Find matching symbols
    matches = [
        s for s in all_symbols
        if s["symbol_name"] == method_name
    ]

    if len(matches) == 1:
        s = matches[0]
        prefix = f"{s['class_name']}." if s['class_name'] else ""
        return f"{prefix}{s['symbol_name']}"

    # Multiple matches: try to narrow by class name hint in the call chain
    if len(parts) >= 2:
        class_hint = parts[-2]
        for s in matches:
            if s.get("class_name", "").lower() == class_hint.lower():
                return f"{s['class_name']}.{s['symbol_name']}"

    if matches:
        s = matches[0]
        prefix = f"{s['class_name']}." if s['class_name'] else ""
        return f"{prefix}{s['symbol_name']}"

    return None

In [ ]:
import json

# Define the skills as OpenAI function-calling tools
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_code",
            "description": "Hybrid semantic + keyword search over the codebase. Returns matching functions with source code.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search query (natural language or identifier names)"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read source code from a file in the repository.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "File path relative to repo root"},
                    "start_line": {"type": "integer", "description": "Start line (1-based, optional)"},
                    "end_line": {"type": "integer", "description": "End line (1-based, optional)"}
                },
                "required": ["path"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "think",
            "description": "Extended reasoning scratchpad. Use to reason about findings before taking action.",
            "parameters": {
                "type": "object",
                "properties": {
                    "reasoning": {"type": "string", "description": "Your reasoning and analysis"}
                },
                "required": ["reasoning"]
            }
        }
    }
]

In [ ]:
def build_dependency_graph(repo_files: dict, all_symbols: list[dict]) -> list[dict]:
    """Build a dependency graph from all files in the repo."""
    edges = []

    for file_path, source in repo_files.items():
        tree = parser.parse(source.encode())
        for node in _walk(tree.root_node):
            if node.type != "function_definition":
                continue

            name_node = node.child_by_field_name("name")
            func_name = name_node.text.decode()
            class_name = _get_class_for_method(node)
            from_symbol = f"{class_name}.{func_name}" if class_name else func_name

            raw_calls = extract_calls(node, file_path)
            for raw in raw_calls:
                resolved = resolve_call(raw, all_symbols)
                if resolved and resolved != from_symbol:
                    edges.append({
                        "from_symbol": from_symbol,
                        "to_symbol": resolved,
                        "dep_type": "calls",
                        "file_path": file_path,
                        "line": node.start_point[0] + 1,
                    })

    return edges


dep_graph = build_dependency_graph(SAMPLE_REPO, all_symbols)
print(f"Dependency graph: {len(dep_graph)} edges\n")
for e in dep_graph:
    print(f"  {e['from_symbol']}  --calls-->  {e['to_symbol']}  ({e['file_path']}:{e['line']})")

In [ ]:
def execute_skill(name: str, args: dict, chunks: list[dict]) -> str:
    """Execute a skill and return the result as a string."""
    if name == "search_code":
        results = hybrid_search(args["query"], chunks, top_k=5)
        output = []
        for r in results:
            prefix = f"{r['class_name']}." if r['class_name'] else ""
            output.append(
                f"--- {prefix}{r['symbol_name']} ({r['file_path']}:{r['start_line']}-{r['end_line']}) ---\n"
                f"{r['full_source']}"
            )
        return "\n\n".join(output) if output else "No results found."

    elif name == "read_file":
        path = args["path"]
        if path not in SAMPLE_REPO:
            return f"File not found: {path}"
        lines = SAMPLE_REPO[path].split("\n")
        start = args.get("start_line", 1) - 1
        end = args.get("end_line", len(lines))
        numbered = [
            f"{i+start+1:4d} | {line}"
            for i, line in enumerate(lines[start:end])
        ]
        return "\n".join(numbered)

    elif name == "think":
        return f"[Reasoning logged: {args['reasoning'][:100]}...]"

    return f"Unknown skill: {name}"

In [ ]:
def get_callers(symbol_name: str, graph: list[dict]) -> list[dict]:
    """Find all symbols that call the given symbol."""
    return [
        {"caller": e["from_symbol"], "file": e["file_path"], "line": e["line"]}
        for e in graph
        if e["to_symbol"] == symbol_name
    ]


def get_dependencies(symbol_name: str, graph: list[dict]) -> list[dict]:
    """Find all symbols that the given symbol calls."""
    return [
        {"callee": e["to_symbol"], "file": e["file_path"], "line": e["line"]}
        for e in graph
        if e["from_symbol"] == symbol_name
    ]


# Who calls get_profile?
callers = get_callers("UserService.get_profile", dep_graph)
print("Callers of UserService.get_profile:")
for c in callers:
    print(f"  {c['caller']}  ({c['file']}:{c['line']})")

# What does get_profile depend on?
deps = get_dependencies("UserService.get_profile", dep_graph)
print(f"\nDependencies of UserService.get_profile:")
for d in deps:
    print(f"  {d['callee']}  ({d['file']}:{d['line']})")

In [ ]:
SYSTEM_PROMPT = """You are a senior software engineer who autonomously investigates
and diagnoses software bugs. You operate inside an agentic harness with access
to a codebase search and file reader.

YOUR TASK:
Given a bug ticket, you will:
1. Understand the bug thoroughly before proposing a fix
2. Locate the exact root cause in the codebase
3. Identify all affected files and callers
4. Propose a minimal, correct fix

METHODOLOGY:
- Use think() to reason explicitly before making decisions.
- Use search_code() to find relevant code. Try both natural language and identifier queries.
- Use read_file() to read full file context when needed.
- Explore before you diagnose. Read broadly, conclude narrowly.

When you have enough information, stop calling tools and write your diagnosis:
- Root cause (file, line, what's wrong)
- Affected files and callers
- Proposed fix (specific code changes)
"""


def run_agent(ticket: dict, chunks: list[dict], max_steps: int = 10):
    """Run the ReAct agent loop on a bug ticket."""
    ticket_text = f"""TICKET: {ticket['ticket_id']}
Title: {ticket['title']}
Description: {ticket['description']}
Severity: {ticket['severity']}
Repo: {ticket['repo']}"""

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": ticket_text},
    ]

    step = 0
    while step < max_steps:
        response = openai.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            tools=TOOLS,
            temperature=0.1,
        )

        msg = response.choices[0].message
        messages.append(msg)

        # If the model produced text without tool calls, it's the diagnosis
        if not msg.tool_calls:
            print(f"\n{'='*60}")
            print("DIAGNOSIS")
            print(f"{'='*60}")
            print(msg.content)
            return msg.content

        # Execute each tool call
        for tc in msg.tool_calls:
            step += 1
            args = json.loads(tc.function.arguments)
            print(f"\nSTEP {step}: {tc.function.name}({json.dumps(args, indent=None)[:120]})")

            result = execute_skill(tc.function.name, args, chunks)
            print(f"  → {result[:200]}{'...' if len(result) > 200 else ''}")

            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result,
            })

    print("\n[Agent hit max steps without producing a diagnosis]")
    return None

### Run the Agent on a Sample Ticket

Our sample repo has a bug: `UserService.get_profile()` can return `None` on a cache miss, but `UserController.get_user()` calls `.to_dict()` on the result without checking. Let's see if the agent finds it.

In [ ]:
sample_ticket = {
    "ticket_id": "PROJ-42",
    "title": "AttributeError in UserController.get_user() when user not in cache",
    "description": (
        "Getting 'AttributeError: NoneType object has no attribute to_dict' "
        "when requesting a user profile that isn't in the cache. "
        "Stack trace points to user_controller.py line 13: profile.to_dict()"
    ),
    "severity": "P2",
    "repo": "github.com/org/user-service",
}

if "OPENAI_API_KEY" in os.environ:
    diagnosis = run_agent(sample_ticket, all_chunks)
else:
    print("Skipped: set OPENAI_API_KEY to run the agent")

## What We Built

Three building blocks of the AI bug-fix pipeline:

1. **Code indexing**: tree-sitter parses source into AST, extracts functions as structured records, chunks at symbol boundaries
2. **Hybrid retrieval**: vector search (enriched embeddings) + BM25 (identifier matching) fused with RRF
3. **ReAct agent loop**: processes a ticket using search and file reading skills, produces a diagnosis

The gap to a production system is operational: Kafka queuing, sandboxed execution for edits and tests, webhook-based incremental indexing, and a persistence layer for sessions and checkpoints. The intelligence is here.